# Trabajo práctico 2: Detección de spam con aprendizaje supervisado y no supervisado

> Notebook principal del proyecto. Cada punto del enunciado se implementa en su propio archivo de `src/tp2/` (`punto1.py`, `punto2.py` y `punto3.py`); este notebook los importa, los combina y presenta los resultados.

**Equipo:** Alejandro Quesada, Manfred Azofeifa, Jose Adrián Herrera

**Estado:** esqueleto. La infraestructura compartida de `utils.py` está lista (carga del dataset, particiones, métricas, repetición de 30 corridas, exportación de figuras y tablas, pruebas). Los tres puntos están pendientes; ver `TODO.md` en la raíz del repositorio y la lista al final del notebook.

## Objetivo y protocolo

**Objetivo.** Detectar mensajes spam del dataset SMSSpamCollection (UCI) con tres enfoques: (1) regresión logística en PyTorch sobre las características de un codificador, con y sin reducción de dimensionalidad por PCA; (2) detección de anomalías con densidades por histograma y un umbral de verosimilitud; y (3) modelos grandes de lenguaje.

**Protocolo común.**

- Etiquetas: `ham = 0` y `spam = 1`. Spam es la clase positiva del F1-score y la anomalía del punto 2.
- Métricas: tasa de aciertos y F1-score (`utils.classification_metrics`).
- Puntos 1.4, 1.5 y 2.2: 30 particiones estratificadas de 70 % entrenamiento y 30 % prueba; la corrida $k$ usa la semilla `SEED + k - 1`.
- Punto 3.1: 30 muestras estratificadas del 30 % del dataset, con las mismas semillas.
- Cada figura mostrada con `utils.show_figure` se guarda en `results/figures/` y cada tabla mostrada con `utils.display_table` se guarda en `results/tables/`.
- TODO: registrar aquí las decisiones del equipo (codificador, D óptima, criterio del umbral, modelos de lenguaje).

## Mapa de archivos

| Punto | Archivo (`src/tp2/`) | Pruebas (`tests/`) | Responsable |
|---|---|---|---|
| 1. Regresión logística (1.1 a 1.5) | `punto1.py` | `test_punto1.py` | por asignar |
| 2. Detección de anomalías (2.1 y 2.2) | `punto2.py` | `test_punto2.py` | por asignar |
| 3. Modelos de lenguaje (3.1 a 3.3) | `punto3.py` | `test_punto3.py` | por asignar |
| Datos, métricas, corridas, figuras y tablas | `utils.py` | `test_utils.py` | listo |

Ningún archivo de `src/tp2/` importa a otro (`tests/test_module_independence.py` lo verifica). Este notebook es el único lugar donde se combinan.

## Configuración y reproducibilidad

Esta sección concentra los imports, las semillas, el protocolo de evaluación y las decisiones compartidas del equipo. Los archivos de cada punto reciben estos valores como argumentos; ninguno lee variables globales del notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch

# El notebook puede abrirse desde la raíz o desde notebooks/: se busca la carpeta src/tp2.
REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "tp2").is_dir())
for module_path in (REPO_ROOT / "src", REPO_ROOT / "tests"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from tp2 import punto1, punto2, punto3, utils

# Reproducibilidad
SEED = 2026
utils.set_global_seed(SEED)

# Protocolo de evaluación
N_RUNS = 30
TRAIN_FRACTION = 0.7
LLM_SAMPLE_FRACTION = 0.3

# Decisiones del equipo (TODO: completar al implementar cada punto)
ENCODER_NAME = None  # Punto 1.4: identificador del codificador
PCA_FRACTIONS = {"50 %": 0.5, "25 %": 0.25}  # Punto 1.5, además de 2 dimensiones
OPTIMAL_PCA_DIMENSIONS = None  # Punto 2.1: D óptima deducida en el punto 1.5
HISTOGRAM_BINS = 30  # Punto 2.1
LLM_MODELS = []  # Punto 3.1 (al menos uno) y 3.2 (extra, al menos dos más)
LORA_MODEL = None  # Punto 3.3 (extra): modelo base que se ajusta con LoRA o DoRA

# Hiperparámetros de la regresión logística (TODO: calibrar)
LR_LEARNING_RATE = 0.1
LR_EPOCHS = 1000

# Rutas: datos descargados y cachés (excluidos de Git) y resultados para el informe
DATA_DIRECTORY = REPO_ROOT / "data" / "raw"
PROCESSED_DIRECTORY = REPO_ROOT / "data" / "processed"
FIGURES_DIRECTORY = utils.configure_figure_export(REPO_ROOT / "results" / "figures")
TABLES_DIRECTORY = utils.configure_table_export(REPO_ROOT / "results" / "tables")

## Convención de documentación interna

Documentar cada función con bloques compatibles con Doxygen. Ejemplo genérico:

```python
def metodo(entrada):
    """!
    @brief Descripción breve.
    @param entrada Descripción del parámetro.
    @return Descripción del resultado.
    """
```

Las funciones de graficación de cada punto devuelven la figura sin mostrarla; este notebook la muestra con `utils.show_figure(...)`, que además la exporta con un nombre derivado de su título.

## Carga del dataset

El archivo se descarga del repositorio UCI la primera vez y se guarda en `data/raw/`. Cada línea contiene la etiqueta (`ham` o `spam`) y el mensaje original; los textos sin preprocesar se usan directamente en el punto 3.

In [ ]:
dataset_path = utils.download_sms_spam(DATA_DIRECTORY)
messages = utils.load_sms_spam(dataset_path)
texts = messages["text"].tolist()
targets = torch.tensor(messages["target"].to_numpy())

class_counts = messages["label"].value_counts().rename_axis("clase").to_frame("mensajes")
class_counts["proporción"] = class_counts["mensajes"] / len(messages)
utils.display_table(class_counts, "Distribución de clases del dataset", formats={"proporción": "{:.2%}"})

---
## 1. Aprendizaje supervisado: regresión logística

Archivo: `src/tp2/punto1.py` · Pruebas: `tests/test_punto1.py` · Informe: `docs/report/sections/01-punto1-regresion-logistica.tex`

### 1.1 y 1.2 Evaluación y entrenamiento

`punto1.forward` evalúa $p(y=1\mid x)=\sigma(x^\top w+b)$ y `punto1.train_logistic_regression` ajusta $w$ y $b$ minimizando la entropía cruzada binaria. Sus pruebas unitarias (1.1a y 1.2a) están en `tests/test_punto1.py` y se ejecutan en la sección **Pruebas unitarias**.

TODO: describir la implementación (inicialización, pérdida, optimizador y criterio de parada).

### 1.3 Datos sintéticos linealmente separables

Se genera un conjunto en 2 dimensiones separable por una recta y se verifica que el regresor logístico encuentra una frontera que lo separa.

In [ ]:
X_synthetic, y_synthetic = punto1.make_linearly_separable(n_samples=200, seed=SEED)
synthetic_model = punto1.train_logistic_regression(
    X_synthetic, y_synthetic, learning_rate=LR_LEARNING_RATE, epochs=LR_EPOCHS, seed=SEED
)
synthetic_predictions = punto1.predict(synthetic_model, X_synthetic)

utils.show_figure(punto1.plot_dataset(X_synthetic, y_synthetic))
utils.show_figure(
    punto1.plot_loss_history(
        synthetic_model, title="Pérdida de la regresión logística en datos sintéticos"
    )
)
utils.show_figure(
    punto1.plot_decision_boundary(
        X_synthetic,
        y_synthetic,
        lambda X: punto1.predict_proba(synthetic_model, X),
        title="Frontera de decisión en datos sintéticos",
    )
)
utils.display_table(
    pd.DataFrame(
        [utils.classification_metrics(y_synthetic, synthetic_predictions)], index=["datos sintéticos"]
    ),
    "Métricas de la regresión logística en datos sintéticos",
    formats="{:.4f}",
)

**Análisis.** TODO: comentar la frontera, la curva de pérdida y las métricas.

### 1.4 Extracción de características y 30 corridas

El codificador convierte cada mensaje en un vector de dimensión $D$. Las características no dependen de la partición, así que se calculan una sola vez y se guardan en `data/processed/`.

In [ ]:
features_path = PROCESSED_DIRECTORY / f"features-{str(ENCODER_NAME).replace('/', '_')}.pt"
if features_path.exists():
    features = torch.load(features_path)
else:
    encoder = punto1.load_encoder(ENCODER_NAME)
    features = punto1.extract_features(texts, encoder)
    PROCESSED_DIRECTORY.mkdir(parents=True, exist_ok=True)
    torch.save(features, features_path)

print(f"Características: {tuple(features.shape)}")

In [ ]:
def logistic_regression_run(seed, n_components=None):
    """!
    @brief Ejecuta una corrida: partición 70/30, PCA opcional, entrenamiento y evaluación.
    @param seed Semilla de la partición y de la inicialización del modelo.
    @param n_components Componentes de PCA; None usa las características originales.
    @return Diccionario con la tasa de aciertos y el F1-score sobre la prueba.
    """
    train_indices, test_indices = utils.stratified_split(targets, TRAIN_FRACTION, seed)
    X_train, X_test = features[train_indices], features[test_indices]
    if n_components is not None:
        # El PCA se ajusta solo con el entrenamiento para no filtrar información de la prueba.
        pca_model = punto1.fit_pca(X_train, n_components)
        X_train = punto1.transform_pca(pca_model, X_train)
        X_test = punto1.transform_pca(pca_model, X_test)

    model = punto1.train_logistic_regression(
        X_train, targets[train_indices], learning_rate=LR_LEARNING_RATE, epochs=LR_EPOCHS, seed=seed
    )
    predictions = punto1.predict(model, X_test)
    return utils.classification_metrics(targets[test_indices], predictions)


classification_results = {
    f"RL (D = {features.shape[1]})": utils.repeat_experiment(
        logistic_regression_run, n_runs=N_RUNS, base_seed=SEED
    )
}
utils.display_table(
    utils.summarize_runs(next(iter(classification_results.values()))),
    "Regresión logística con las características del codificador (30 corridas)",
    formats="{:.4f}",
)

**Análisis.** TODO: comentar la media y la dispersión de las métricas en las 30 corridas.

### 1.5 Reducción de dimensionalidad con PCA

Se reduce a 50 % y 25 % de $D$ y a 2 dimensiones. En cada corrida el PCA se ajusta con el entrenamiento; para graficar todo el conjunto de datos se ajusta un PCA sobre todos los mensajes.

In [ ]:
pca_dimensions = {
    f"{label} de D": punto1.components_for_fraction(features.shape[1], fraction)
    for label, fraction in PCA_FRACTIONS.items()
}
pca_dimensions["2 dimensiones"] = 2

full_pca = punto1.fit_pca(features, max(pca_dimensions.values()))
utils.show_figure(punto1.plot_explained_variance(full_pca))
for name, n_components in pca_dimensions.items():
    projection = punto1.transform_pca(punto1.fit_pca(features, n_components), features)
    utils.show_figure(
        punto1.plot_pca_projection(projection, targets, title=f"Proyección PCA del dataset: {name}")
    )

In [ ]:
for name, n_components in pca_dimensions.items():
    classification_results[f"RL + PCA {name} (k = {n_components})"] = utils.repeat_experiment(
        lambda seed, k=n_components: logistic_regression_run(seed, n_components=k),
        n_runs=N_RUNS,
        base_seed=SEED,
    )

utils.display_table(
    utils.compare_experiments(classification_results),
    "Regresión logística con y sin PCA (30 corridas)",
    formats="{:.4f}",
)
utils.show_figure(
    utils.plot_metric_distributions(
        classification_results, "f1", title="F1-score de la regresión logística con y sin PCA"
    )
)

**Análisis.** TODO: comparar las dimensiones, deducir la $D$ óptima y fijarla en `OPTIMAL_PCA_DIMENSIONS` (celda de configuración).

---
## 2. Detección de anomalías: umbral de verosimilitud

Archivo: `src/tp2/punto2.py` · Pruebas: `tests/test_punto2.py` · Informe: `docs/report/sections/02-punto2-deteccion-anomalias.tex`

### 2.1 Densidades por histograma

Con la $D$ óptima del punto 1.5, cada dimensión se modela con un histograma $p_d$ y el puntaje de un mensaje es $\sum_{d=1}^{D}\log p_d(x_d)$. Un puntaje menor que el umbral marca el mensaje como fuera de la distribución (spam).

TODO: justificar qué datos de entrenamiento construyen las densidades y el criterio del umbral.

In [ ]:
def anomaly_detection_split(seed):
    """!
    @brief Ajusta PCA y densidades con el entrenamiento de una partición y puntúa la prueba.
    @param seed Semilla de la partición.
    @return Diccionario con las densidades, el umbral, los puntajes y las etiquetas de prueba.
    """
    if OPTIMAL_PCA_DIMENSIONS is None:
        raise NotImplementedError("Punto 2.1: fijar OPTIMAL_PCA_DIMENSIONS en la configuración.")

    train_indices, test_indices = utils.stratified_split(targets, TRAIN_FRACTION, seed)
    pca_model = punto1.fit_pca(features[train_indices], OPTIMAL_PCA_DIMENSIONS)
    X_train = punto1.transform_pca(pca_model, features[train_indices])
    X_test = punto1.transform_pca(pca_model, features[test_indices])

    # TODO(punto 2.1): confirmar la decisión; aquí las densidades se construyen solo con ham.
    normal_train = X_train[targets[train_indices] == 0]
    densities = punto2.fit_histogram_densities(normal_train, n_bins=HISTOGRAM_BINS)
    train_scores = punto2.log_likelihood(densities, normal_train)
    return {
        "densities": densities,
        "threshold": punto2.select_threshold(train_scores),
        "test_scores": punto2.log_likelihood(densities, X_test),
        "test_targets": targets[test_indices],
    }


anomaly_demo = anomaly_detection_split(SEED)
utils.show_figure(punto2.plot_densities(anomaly_demo["densities"]))
utils.show_figure(
    punto2.plot_score_distribution(
        anomaly_demo["test_scores"], anomaly_demo["test_targets"], anomaly_demo["threshold"]
    )
)

**Análisis.** TODO: comentar la forma de las densidades y la separación de puntajes entre clases.

### 2.2 Evaluación en 30 corridas

In [ ]:
def anomaly_detection_run(seed):
    """!
    @brief Ejecuta una corrida del detector de anomalías y evalúa la prueba.
    @param seed Semilla de la partición.
    @return Diccionario con la tasa de aciertos y el F1-score sobre la prueba.
    """
    split = anomaly_detection_split(seed)
    predictions = punto2.predict_anomalies(split["test_scores"], split["threshold"])
    return utils.classification_metrics(split["test_targets"], predictions)


anomaly_results = {
    "Densidades por histograma": utils.repeat_experiment(
        anomaly_detection_run, n_runs=N_RUNS, base_seed=SEED
    )
}
utils.display_table(
    utils.compare_experiments({**classification_results, **anomaly_results}),
    "Detección de anomalías frente a la regresión logística (30 corridas)",
    formats="{:.4f}",
)

**Análisis.** TODO: comparar contra los resultados supervisados del punto 1.

---
## 3. Clasificación supervisada: modelos grandes de lenguaje

Archivo: `src/tp2/punto3.py` · Pruebas: `tests/test_punto3.py` · Informe: `docs/report/sections/03-punto3-modelos-lenguaje.tex`

### 3.1 y 3.2 Clasificación con modelos de lenguaje

Con un modelo fijo, la predicción de cada mensaje no depende de la muestra. Por eso cada mensaje se clasifica una sola vez por modelo (las predicciones se guardan en `data/processed/`) y las 30 muestras del 30 % reutilizan esas predicciones: el resultado es el mismo que clasificar cada muestra, con muchas menos inferencias.

In [ ]:
if not LLM_MODELS:
    raise NotImplementedError("Punto 3.1: definir LLM_MODELS en la configuración.")

llm_predictions = {}
for model_name in LLM_MODELS:
    predictions_path = PROCESSED_DIRECTORY / f"llm-{model_name.replace('/', '_')}.pt"
    if predictions_path.exists():
        llm_predictions[model_name] = torch.load(predictions_path)
    else:
        llm = punto3.load_llm(model_name)
        llm_predictions[model_name] = punto3.classify_messages(texts, llm)
        PROCESSED_DIRECTORY.mkdir(parents=True, exist_ok=True)
        torch.save(llm_predictions[model_name], predictions_path)

In [ ]:
def llm_run(seed, predictions):
    """!
    @brief Evalúa las predicciones de un modelo sobre una muestra estratificada del 30 %.
    @param seed Semilla del muestreo.
    @param predictions Predicciones del modelo para todos los mensajes, forma (n,).
    @return Diccionario con la tasa de aciertos y el F1-score sobre la muestra.
    """
    sampled = utils.sample_indices(targets, LLM_SAMPLE_FRACTION, seed)
    return utils.classification_metrics(targets[sampled], predictions[sampled])


llm_results = {
    model_name: utils.repeat_experiment(
        lambda seed, p=predictions: llm_run(seed, p), n_runs=N_RUNS, base_seed=SEED
    )
    for model_name, predictions in llm_predictions.items()
}
utils.display_table(
    utils.compare_experiments(llm_results),
    "Clasificación con modelos de lenguaje (30 muestras del 30 %)",
    formats="{:.4f}",
)

**Análisis.** TODO: comparar los modelos de lenguaje entre sí y con los resultados anteriores.

### 3.3 (extra) Ajuste fino eficiente con LoRA o DoRA

In [ ]:
if LORA_MODEL is None:
    print("Punto 3.3 (extra) sin configurar: se omite. Definir LORA_MODEL para ejecutarlo.")
else:
    train_indices, test_indices = utils.stratified_split(targets, TRAIN_FRACTION, SEED)
    lora_model = punto3.finetune_with_lora(
        LORA_MODEL,
        [texts[index] for index in train_indices.tolist()],
        targets[train_indices],
        seed=SEED,
    )
    lora_predictions = punto3.classify_finetuned(
        lora_model, [texts[index] for index in test_indices.tolist()]
    )
    utils.display_table(
        pd.DataFrame(
            [utils.classification_metrics(targets[test_indices], lora_predictions)],
            index=[f"{LORA_MODEL} + LoRA"],
        ),
        "Modelo de lenguaje ajustado con LoRA",
        formats="{:.4f}",
    )

---
## Comparación de resultados

Todas las filas resumen 30 corridas. Los enfoques 1 y 2 se evalúan en el 30 % de prueba de cada partición; los modelos de lenguaje, en muestras del 30 % de todo el dataset.

In [ ]:
all_results = {**classification_results, **anomaly_results, **llm_results}
utils.display_table(
    utils.compare_experiments(all_results), "Comparación de todos los métodos", formats="{:.4f}"
)
for metric_name, metric_label in (("tasa_aciertos", "tasa de aciertos"), ("f1", "F1-score")):
    utils.show_figure(
        utils.plot_metric_distributions(
            all_results, metric_name, title=f"Comparación de métodos: {metric_label}"
        )
    )

---
## Pruebas unitarias

Las pruebas viven en `tests/`, un archivo por punto más `test_utils.py`, y también se ejecutan con `pytest` desde la raíz del repositorio. Las pruebas sin implementar se reportan como pendientes. Esta celda solo necesita la celda de configuración, así que se puede ejecutar en cualquier momento para ver el avance.

In [ ]:
import test_module_independence
import test_punto1
import test_punto2
import test_punto3
import test_utils

unit_test_results = utils.run_tests(
    test_module_independence, test_utils, test_punto1, test_punto2, test_punto3
)

## Conclusiones y pendientes

Las respuestas, el análisis y las conclusiones se desarrollan en el informe LaTeX (`docs/report/`), con los resultados de este notebook. El detalle de cada tarea, las restricciones y la lista de entrega están en `TODO.md`.

- [ ] 1.1: `forward` y sus dos pruebas unitarias.
- [ ] 1.2: `train_logistic_regression` y sus dos pruebas unitarias.
- [ ] 1.3: datos sintéticos, gráficos y tablas.
- [ ] 1.4: codificador (`ENCODER_NAME`) y 30 corridas.
- [ ] 1.5: PCA a 50 %, 25 % y 2 dimensiones; fijar `OPTIMAL_PCA_DIMENSIONS`.
- [ ] 2.1: densidades por histograma y umbral.
- [ ] 2.2: 30 corridas del detector de anomalías.
- [ ] 3.1: al menos un modelo de lenguaje (`LLM_MODELS`).
- [ ] 3.2 (extra): al menos dos modelos más.
- [ ] 3.3 (extra): ajuste fino con LoRA o DoRA (`LORA_MODEL`).
- [ ] Completar el informe LaTeX y generar el PDF.
- [ ] Generar la documentación interna con Doxygen (ver README).
- [ ] Resumir aquí, en pocas líneas, las conclusiones del informe.